<center>
  <h1><b>SeedUp - Smart Torrent Management V1</b></h1>
  
  <a href="https://www.buymeacoffee.com/codercyco" target="_blank">
    <img src="https://cdn.buymeacoffee.com/buttons/v2/default-blue.png" alt="Buy Me A Coffee" width="300">
  </a>
  
  <br>
  
  ### 🌐 **Connect & Stay Updated**
  <!-- Social Media Links -->
  <div style="display: flex; justify-content: center; align-items: center; gap: 10px; flex-wrap: wrap;">
    <a href="https://www.linkedin.com/in/isharadeshapriya/" target="_blank">
      <img src="https://img.shields.io/badge/Follow-LinkedIn-0077B5?style=for-the-badge&logo=linkedin&logoColor=white" alt="Follow on LinkedIn" height="30">
    </a>
    <a href="https://www.linkedin.com/newsletters/7355638830797901825/" target="_blank">
      <img src="https://img.shields.io/badge/Subscribe-Newsletter-0077B5?style=for-the-badge&logo=linkedin&logoColor=white" alt="Subscribe to Newsletter" height="30">
    </a>
    <a href="https://www.youtube.com/@0xbashbyte" target="_blank">
      <img src="https://img.shields.io/badge/Subscribe-YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" alt="Subscribe on YouTube" height="30">
    </a>
  </div>
</center>

SeedUp is a Python based tool that combines torrent downloading with VikingFile uploading capabilities. It's designed to work both as a standalone application and in Google Colab environments, making it perfect for managing downloads without using local resources.

---


# **Features**

This notebook provides:

* ⚡ High-speed torrent downloading using libtorrent
* ☁️ Automatic upload to VikingFile
* 📁 **Organized uploads** into a configurable VikingFile folder path
* ⏸️ Resume capability for interrupted downloads
* 📊 Real-time progress tracking
* 🔄 Skip already uploaded files
* 🎯 Support for both magnet links and .torrent files

---

### **Important Notes:**

1. **Free Tier Limitations:** Google Colab free tier has:
   - Limited runtime
   - Limited disk space (~100GB)
   - Session may disconnect if idle

2. **Legal Usage Only:** Only download content you have the right to download.

3. **VikingFile Storage:** VikingFile offers unlimited free storage per file.

---

### **Security & Privacy Notes:**

1. **VikingFile Account:**
   - Uploads are associated with your account via a user hash (no OAuth/login flow)
   - Leave the user hash blank to upload anonymously

2. **Data Privacy:**
   - All downloads are temporary and deleted after upload
   - No logs are kept of your download history
   - Session data is cleared automatically

3. **Responsible Usage:**
   - Only download content you have legal rights to access
   - Respect copyright and intellectual property laws
   - Be aware of your local regulations regarding torrents

<br>

---
# **📥 Step 1: Create Project Files**

Write the torrent downloader and VikingFile uploader scripts into this Colab session.

This writes the SeedUp script files directly into the Colab environment below — no manual upload or git clone needed. Just run each cell in order.

In [ ]:
%%writefile config.py
"""
SeedUp - Smart Torrent Management Tool
Shared configuration and constants for the torrent downloader and VikingFile uploader.

Copyright 2025 Ishara Deshapriya

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
"""

import os
import json
import logging

# Torrent Downloader Configuration
TORRENT_SESSION_FILE = "torrent_session.json"
TORRENT_DOWNLOAD_PATH = "../SeedUp Downloads"

# VikingFile Uploader Configuration
VIKINGFILE_API_BASE = "https://vikingfile.com/api"

# Default user hash used to associate uploads with a VikingFile account.
# Leave as an empty string (or pass user=None / --anonymous on the CLI) to
# upload anonymously instead.
VIKINGFILE_USER_HASH = "ndJCSIGAsT"

MAX_RETRIES = 15
RETRY_DELAY = 2  # seconds base delay for exponential backoff
MAX_RETRY_DELAY = 60  # cap on the exponential backoff, regardless of attempt number
LARGE_FILE_THRESHOLD = 1024 * 1024 * 1024  # 1GB
PROGRESS_FILE = '.vikingfile_upload_progress.json'
CONFIG_FILE = '.vikingfile-uploader.conf'

# Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def get_logger(name):
    """Get a logger instance."""
    return logging.getLogger(name)


class ConfigManager:
    """Manage configuration file for storing default settings."""

    @staticmethod
    def load_config(config_path: str = CONFIG_FILE) -> dict:
        """Load configuration from file."""
        logger = get_logger(__name__)
        if os.path.exists(config_path):
            try:
                with open(config_path, 'r') as f:
                    return json.load(f)
            except Exception as e:
                logger.warning(f"Could not load config file: {e}")
        return {}

    @staticmethod
    def save_config(config: dict, config_path: str = CONFIG_FILE):
        """Save configuration to file."""
        logger = get_logger(__name__)
        try:
            with open(config_path, 'w') as f:
                json.dump(config, f, indent=2)
            logger.info(f"Configuration saved to {config_path}")
        except Exception as e:
            logger.error(f"Could not save config file: {e}")


In [ ]:
%%writefile torrent_downloader.py
"""
SeedUp - Smart Torrent Management Tool
Torrent downloader module using libtorrent with resume capability.

Copyright 2025 Ishara Deshapriya

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
"""

import libtorrent as lt
import time
import os
import sys
from config import TORRENT_SESSION_FILE, TORRENT_DOWNLOAD_PATH, get_logger

logger = get_logger(__name__)

# Check if running in Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Note: this module runs as a subprocess (invoked via `!python main.py ...`
# from Colab), not as code inside the notebook's own kernel — so
# IPython.display.clear_output() is not usable here; it only works for code
# executing directly in the kernel. Multi-line progress redraws instead use
# raw ANSI cursor-movement codes written straight to stdout (see
# download_torrents() below), the same mechanism a real terminal uses.


def save_session(session, session_file=TORRENT_SESSION_FILE):
    """Save session state to resume later (correctly saves binary data)."""
    try:
        with open(session_file, "wb") as f:
            session_state = session.save_state()
            f.write(lt.bencode(session_state))
        logger.debug(f"Session saved to {session_file}")
    except Exception as e:
        logger.error(f"Failed to save session: {e}")


def load_session(session_file=TORRENT_SESSION_FILE):
    """Load session state if exists, otherwise return a new session."""
    if os.path.exists(session_file):
        try:
            with open(session_file, "rb") as f:
                session_data = f.read()
                if not session_data:
                    raise ValueError("Session file is empty.")
                
                session_state = lt.bdecode(session_data)
                ses = lt.session()
                ses.load_state(session_state)
                logger.info(f"Session loaded from {session_file}")
                return ses
        except (RuntimeError, ValueError) as e:
            logger.warning(f"Failed to load session ({e}). Starting fresh.")
            os.remove(session_file)
    
    return lt.session()


def list_torrent_files(source, timeout=60):
    """
    Fetch a torrent's file list (index, path, size) without downloading any
    of the actual file data. For magnet links this briefly connects to
    peers/DHT just long enough to retrieve metadata.

    :param source: .torrent file path or magnet link.
    :param timeout: Max seconds to wait for metadata (magnet links only).
    :return: List of dicts: {'index': int, 'path': str, 'size': int} or None on failure.
    """
    ses = lt.session()
    ses.apply_settings({'listen_interfaces': '0.0.0.0:6881'})

    params = lt.add_torrent_params()
    params.save_path = "."  # not used, no data is downloaded
    # Don't download any data yet, just metadata.
    params.flags |= lt.torrent_flags.upload_mode

    if source.startswith("magnet:"):
        params.url = source
    elif source.endswith(".torrent"):
        if not os.path.exists(source):
            logger.error(f"Torrent file not found: {source}")
            return None
        try:
            with open(source, "rb") as f:
                torrent_data = lt.bdecode(f.read())
                info = lt.torrent_info(torrent_data)
                params.ti = info
        except Exception as e:
            logger.error(f"Failed to read torrent file: {e}")
            return None
    else:
        logger.error("Invalid source. Provide a .torrent file or magnet link.")
        return None

    try:
        handle = ses.add_torrent(params)
    except Exception as e:
        logger.error(f"Failed to add torrent: {e}")
        return None

    waited = 0
    while not handle.status().has_metadata:
        if waited >= timeout:
            logger.error("Timed out waiting for torrent metadata.")
            ses.remove_torrent(handle)
            return None
        time.sleep(1)
        waited += 1

    info = handle.torrent_file()
    storage = info.files()
    files = []
    for i in range(storage.num_files()):
        files.append({
            "index": i,
            "path": storage.file_path(i),
            "size": storage.file_size(i),
        })

    ses.remove_torrent(handle)
    return files


def download_torrent(source, download_path=TORRENT_DOWNLOAD_PATH, 
                    session_file=TORRENT_SESSION_FILE, auto_resume=True,
                    file_indices=None):
    """
    Download a torrent file using libtorrent, with support for stopping/resuming.
    
    :param source: .torrent file path or magnet link.
    :param download_path: Directory to save the downloaded content.
    :param session_file: File to save/load session state.
    :param auto_resume: Automatically load previous session if available.
    :param file_indices: Optional list of file indices (from list_torrent_files)
                          to download. Files not in this list are skipped
                          entirely (never requested from peers). None downloads
                          every file in the torrent.
    :return: Path to downloaded content or None on failure.
    """
    if not os.path.exists(download_path):
        os.makedirs(download_path)
        logger.info(f"Created download directory: {download_path}")

    # Check if we're resuming from a previous session
    is_resuming = auto_resume and os.path.exists(session_file)

    # Load existing session or create new one
    ses = load_session(session_file) if auto_resume else lt.session()

    # Apply necessary settings
    settings = {
        'listen_interfaces': '0.0.0.0:6881',
    }
    ses.apply_settings(settings)

    # Initialize add_torrent_params
    params = lt.add_torrent_params()
    params.save_path = download_path
    params.storage_mode = lt.storage_mode_t.storage_mode_sparse

    # Handle magnet link or .torrent file
    if source.startswith("magnet:"):
        params.url = source
        logger.info(f"Adding magnet link: {source[:60]}...")
    elif source.endswith(".torrent"):
        if not os.path.exists(source):
            logger.error(f"Torrent file not found: {source}")
            return None
        
        try:
            with open(source, "rb") as f:
                torrent_data = lt.bdecode(f.read())
                info = lt.torrent_info(torrent_data)
                params.ti = info
            logger.info(f"Adding torrent file: {source}")
        except Exception as e:
            logger.error(f"Failed to read torrent file: {e}")
            return None
    else:
        logger.error("Invalid source. Provide a .torrent file or magnet link.")
        return None

    # Add the torrent to the session
    try:
        handle = ses.add_torrent(params)
        logger.info(f"Downloading to: {download_path}")
    except Exception as e:
        logger.error(f"Failed to add torrent: {e}")
        return None

    # Wait for metadata
    logger.info("Waiting for metadata...")
    while not handle.status().has_metadata:
        time.sleep(1)

    torrent_name = handle.status().name
    logger.info(f"Downloading: {torrent_name}")

    # If specific files were requested, skip everything else entirely so
    # unselected files are never fetched from peers in the first place.
    if file_indices is not None:
        info = handle.torrent_file()
        num_files = info.files().num_files()
        selected = set(file_indices)
        priorities = [4 if i in selected else 0 for i in range(num_files)]
        handle.prioritize_files(priorities)
        logger.info(f"Selective download: {len(selected)}/{num_files} file(s) selected")

    try:
        while handle.status().state not in (lt.torrent_status.finished, lt.torrent_status.seeding):
            s = handle.status()
            progress = s.progress * 100

            # Calculate ETA
            eta_str = "N/A"
            if s.download_rate > 0:
                total_size = s.total_wanted
                downloaded = s.total_done
                remaining = total_size - downloaded
                eta_seconds = remaining / s.download_rate
                
                if eta_seconds < 60:
                    eta_str = f"{int(eta_seconds)}s"
                elif eta_seconds < 3600:
                    eta_str = f"{int(eta_seconds / 60)}m {int(eta_seconds % 60)}s"
                else:
                    hours = int(eta_seconds / 3600)
                    minutes = int((eta_seconds % 3600) / 60)
                    eta_str = f"{hours}h {minutes}m"

            # Format download speed
            if s.download_rate > 1024 * 1024:  # > 1 MB/s
                speed_str = f"{s.download_rate / (1024 * 1024):.2f} MB/s"
            else:
                speed_str = f"{s.download_rate / 1024:.2f} KB/s"

            # Build complete progress bar string manually
            bar_length = 30
            filled_length = int(bar_length * progress / 100)
            bar = '█' * filled_length + '░' * (bar_length - filled_length)
            
            # Determine label based on actual state
            if is_resuming and progress < 95:
                label = "Resuming Download"
            elif s.download_rate == 0 and s.num_peers == 0:
                label = "Connecting to Peers"
            else:
                label = "Download Progress"
                is_resuming = False  # No longer resuming once we're actively downloading
            
            stats_str = f"Seeds: {s.num_seeds} | Peers: {s.num_peers - s.num_seeds} | Speed: {speed_str} | ETA: {eta_str}"
            progress_line = f"{label}: {bar} {progress:.1f}/100%    | {stats_str}"
            
            # Use simple print instead of tqdm to avoid interference
            print(f"\r{progress_line}", end="", flush=True)

            # Save session periodically (every 10 seconds)
            if int(time.time()) % 10 == 0:
                save_session(ses, session_file)
            
            time.sleep(1)

    except KeyboardInterrupt:
        print()  # New line after progress bar
        logger.warning("Download paused by user. Session saved for resume.")
        save_session(ses, session_file)
        return None
    
    print()  # New line after progress bar completion

    logger.info("Download complete!")

    # Remove placeholder files/folders for any files that were deselected,
    # so only the files you actually chose end up in the download folder.
    if file_indices is not None:
        info = handle.torrent_file()
        storage = info.files()
        selected = set(file_indices)
        for i in range(storage.num_files()):
            if i in selected:
                continue
            skipped_path = os.path.join(download_path, storage.file_path(i))
            try:
                if os.path.exists(skipped_path):
                    os.remove(skipped_path)
            except OSError as e:
                logger.debug(f"Could not remove skipped placeholder '{skipped_path}': {e}")
        # Clean up any now-empty subdirectories left behind.
        root_dir = os.path.join(download_path, torrent_name)
        for dirpath, dirnames, filenames in os.walk(root_dir, topdown=False):
            if not dirnames and not filenames and dirpath != root_dir:
                try:
                    os.rmdir(dirpath)
                except OSError:
                    pass

    # Clean up session file on successful completion
    if os.path.exists(session_file):
        try:
            os.remove(session_file)
            logger.debug("Session file removed after successful download")
        except Exception as e:
            logger.warning(f"Could not remove session file: {e}")
    
    # Return the path to downloaded content
    downloaded_path = os.path.join(download_path, torrent_name)
    return downloaded_path


def _format_speed(rate):
    """Human-readable download speed from a bytes/sec rate."""
    if rate > 1024 * 1024:
        return f"{rate / (1024 * 1024):.2f} MB/s"
    return f"{rate / 1024:.2f} KB/s"


def _format_eta(seconds):
    if seconds is None or seconds == float('inf'):
        return "N/A"
    if seconds < 60:
        return f"{int(seconds)}s"
    if seconds < 3600:
        return f"{int(seconds / 60)}m {int(seconds % 60)}s"
    hours = int(seconds / 3600)
    minutes = int((seconds % 3600) / 60)
    return f"{hours}h {minutes}m"


def _cleanup_unselected_files(handle, file_indices, result_dir, download_path):
    """Remove placeholder files/folders for files that were deselected, so
    only the chosen files end up in the download folder."""
    if file_indices is None:
        return
    info = handle.torrent_file()
    storage = info.files()
    selected = set(file_indices)
    for i in range(storage.num_files()):
        if i in selected:
            continue
        skipped_path = os.path.join(download_path, storage.file_path(i))
        try:
            if os.path.exists(skipped_path):
                os.remove(skipped_path)
        except OSError as e:
            logger.debug(f"Could not remove skipped placeholder '{skipped_path}': {e}")
    for dirpath, dirnames, filenames in os.walk(result_dir, topdown=False):
        if not dirnames and not filenames and dirpath != result_dir:
            try:
                os.rmdir(dirpath)
            except OSError:
                pass


def download_torrents(sources, download_path=TORRENT_DOWNLOAD_PATH,
                       session_file=TORRENT_SESSION_FILE, auto_resume=True,
                       file_indices_map=None, metadata_timeout=120,
                       on_torrent_complete=None):
    """
    Download multiple torrents concurrently within a single shared libtorrent
    session, with a combined progress display for all of them.

    Running torrents in one shared session (rather than one process per
    torrent) means they cooperatively share the same disk I/O and bandwidth
    accounting, which is the same underlying network/disk pool Colab gives
    you either way — this just avoids spinning up N separate Python
    processes/sessions to do it.

    :param sources: list of magnet links / .torrent file paths.
    :param download_path: Shared base directory; each torrent gets its own
                           subfolder here (named after the torrent), same as
                           a single download would.
    :param session_file: File to save/load combined session state.
    :param auto_resume: Automatically load previous session if available.
    :param file_indices_map: Optional dict {source: file_indices_list} for
                              per-torrent selective downloads. A source
                              that's absent or maps to None downloads every
                              file in that torrent.
    :param metadata_timeout: Max seconds to wait for a magnet link's
                              metadata before giving up on that torrent
                              (other torrents in the batch are unaffected).
    :param on_torrent_complete: Optional callback ``fn(source, downloaded_path)``
                         invoked the instant an individual torrent finishes —
                         while the rest of the batch may still be downloading.
                         Intended for kicking off a per-torrent upload without
                         waiting for the whole batch. May return a string,
                         which is kept as that torrent's permanent status
                         line in the combined progress display (e.g. an
                         upload result), instead of just "complete".
    :return: dict {source: downloaded_path_or_None}. None means that
             torrent failed, timed out, or was still incomplete when the
             batch was interrupted.
    """
    if not sources:
        return {}

    file_indices_map = file_indices_map or {}

    if not os.path.exists(download_path):
        os.makedirs(download_path)
        logger.info(f"Created download directory: {download_path}")

    ses = load_session(session_file) if auto_resume else lt.session()
    ses.apply_settings({'listen_interfaces': '0.0.0.0:6881'})

    # jobs: source -> dict with handle, name, file_indices, result, done, added_ok
    jobs = {}
    for source in sources:
        params = lt.add_torrent_params()
        params.save_path = download_path
        params.storage_mode = lt.storage_mode_t.storage_mode_sparse

        if source.startswith("magnet:"):
            params.url = source
        elif source.endswith(".torrent"):
            if not os.path.exists(source):
                logger.error(f"Torrent file not found, skipping: {source}")
                jobs[source] = {"handle": None, "name": source, "result": None,
                                 "done": True, "file_indices": None, "prioritized": True}
                continue
            try:
                with open(source, "rb") as f:
                    torrent_data = lt.bdecode(f.read())
                    params.ti = lt.torrent_info(torrent_data)
            except Exception as e:
                logger.error(f"Failed to read torrent file '{source}', skipping: {e}")
                jobs[source] = {"handle": None, "name": source, "result": None,
                                 "done": True, "file_indices": None, "prioritized": True}
                continue
        else:
            logger.error(f"Invalid source, skipping: {source}")
            jobs[source] = {"handle": None, "name": source, "result": None,
                             "done": True, "file_indices": None, "prioritized": True}
            continue

        try:
            handle = ses.add_torrent(params)
        except Exception as e:
            logger.error(f"Failed to add torrent '{source}', skipping: {e}")
            jobs[source] = {"handle": None, "name": source, "result": None,
                             "done": True, "file_indices": None, "prioritized": True}
            continue

        jobs[source] = {
            "handle": handle,
            "name": source[:40],
            "result": None,
            "done": False,
            "file_indices": file_indices_map.get(source),
            "prioritized": False,
        }

    active_sources = [s for s, j in jobs.items() if j["handle"] is not None]
    if not active_sources:
        logger.error("No valid torrents to download.")
        return {s: None for s in sources}

    print(f"📦 Added {len(active_sources)} torrent(s) to a shared session\n")

    start_time = time.time()
    PAD_WIDTH = 160  # wide enough that a shorter new line fully overwrites a longer old one

    def _finalize_job(source, job):
        """Clean up deselected placeholder files and fire on_torrent_complete, the
        instant this specific torrent finishes (not waiting on the others)."""
        job["result"] = os.path.join(download_path, job["name"])

        if job["file_indices"] is not None:
            handle = job["handle"]
            info = handle.torrent_file()
            storage = info.files()
            selected = set(job["file_indices"])
            for i in range(storage.num_files()):
                if i in selected:
                    continue
                skipped_path = os.path.join(download_path, storage.file_path(i))
                try:
                    if os.path.exists(skipped_path):
                        os.remove(skipped_path)
                except OSError as e:
                    logger.debug(f"Could not remove skipped placeholder '{skipped_path}': {e}")
            for dirpath, dirnames, filenames in os.walk(job["result"], topdown=False):
                if not dirnames and not filenames and dirpath != job["result"]:
                    try:
                        os.rmdir(dirpath)
                    except OSError:
                        pass

        message = None
        if on_torrent_complete is not None:
            try:
                message = on_torrent_complete(source, job["result"])
            except Exception as e:
                logger.error(f"on_torrent_complete callback failed for '{job['name']}': {e}")
        return message or f"✅ {job['name']}: complete"

    try:
        while True:
            all_done = True
            active_summaries = []
            newly_finished_messages = []

            for source in active_sources:
                job = jobs[source]
                if job["done"]:
                    continue  # already permanently printed, nothing more to do

                handle = job["handle"]
                s = handle.status()

                if not s.has_metadata:
                    all_done = False
                    if time.time() - start_time > metadata_timeout:
                        logger.warning(f"Timed out waiting for metadata: {source[:60]}")
                        job["done"] = True
                        job["result"] = None
                        newly_finished_messages.append(f"❌ {job['name']}: metadata timeout")
                        continue
                    active_summaries.append(f"{job['name'][:20]}: waiting for metadata...")
                    continue

                if job["name"] == source[:40]:
                    job["name"] = s.name  # switch to the real torrent name once known

                # Apply file-selection priorities once, right after metadata arrives.
                if not job["prioritized"]:
                    if job["file_indices"] is not None:
                        info = handle.torrent_file()
                        num_files = info.files().num_files()
                        selected = set(job["file_indices"])
                        priorities = [4 if i in selected else 0 for i in range(num_files)]
                        handle.prioritize_files(priorities)
                        logger.info(f"[{job['name']}] Selective download: {len(selected)}/{num_files} file(s) selected")
                    job["prioritized"] = True

                if s.state in (lt.torrent_status.finished, lt.torrent_status.seeding):
                    job["done"] = True
                    newly_finished_messages.append(_finalize_job(source, job))
                    continue

                all_done = False
                progress = s.progress * 100

                if s.download_rate == 0 and s.num_peers == 0:
                    label = "connecting"
                else:
                    label = _format_speed(s.download_rate)

                active_summaries.append(f"{job['name'][:20]}: {progress:5.1f}% ({label})")

            # Torrents that finished THIS cycle get a permanent line each,
            # appended normally (guaranteed to render correctly, no special
            # terminal support needed). Clear the rolling line first so a
            # stale in-progress percentage doesn't linger above them.
            if newly_finished_messages:
                print("\r" + " " * PAD_WIDTH + "\r", end="")
                for msg in newly_finished_messages:
                    print(msg)

            # The rest of the still-active torrents share ONE continuously
            # overwritten line at the bottom — this is the same \r mechanism
            # the single-torrent progress bar already relies on, just with
            # every active torrent's status packed into that one line
            # (genuine independent multi-line in-place updates aren't
            # reliable from this subprocess's captured stdout).
            if active_summaries:
                line = f"⬇️  [{len(active_summaries)} active] " + " | ".join(active_summaries)
                print("\r" + line[:PAD_WIDTH].ljust(PAD_WIDTH), end="", flush=True)

            if int(time.time()) % 10 == 0:
                save_session(ses, session_file)

            if all_done:
                break

            time.sleep(1)

    except KeyboardInterrupt:
        print()
        logger.warning("Download batch paused by user. Session saved for resume.")
        save_session(ses, session_file)
        return {s: jobs[s]["result"] if s in jobs else None for s in sources}

    print()
    logger.info("All torrents in batch finished downloading!")

    if os.path.exists(session_file):
        try:
            os.remove(session_file)
            logger.debug("Session file removed after successful batch download")
        except Exception as e:
            logger.warning(f"Could not remove session file: {e}")

    return {s: jobs[s]["result"] for s in sources}



def get_download_status(session_file=TORRENT_SESSION_FILE):
    """
    Check if there's a paused download that can be resumed.
    
    :return: True if a session file exists, False otherwise.
    """
    return os.path.exists(session_file)


def clear_session(session_file=TORRENT_SESSION_FILE):
    """
    Clear the session file to start fresh.
    
    :return: True if cleared successfully, False otherwise.
    """
    if os.path.exists(session_file):
        try:
            os.remove(session_file)
            logger.info("Session file cleared")
            return True
        except Exception as e:
            logger.error(f"Failed to clear session: {e}")
            return False
    return True


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python torrent_downloader.py <torrent_file/magnet_link>")
        sys.exit(1)

    source = sys.argv[1]
    result = download_torrent(source)
    
    if result:
        print(f"\nDownloaded to: {result}")
        sys.exit(0)
    else:
        sys.exit(1)

In [ ]:
%%writefile vikingfile_uploader.py
"""
SeedUp - Smart Torrent Management Tool
VikingFile uploader module.

Copyright 2025 Ishara Deshapriya

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

Uses the public VikingFile API (https://vikingfile.com/api) to upload files.
Large files are uploaded using the multi-part upload flow
(get-upload-url -> PUT each part -> complete-upload), which is more
resilient over long-running Colab sessions than a single request.

No authentication is required beyond an (optional) account "user hash",
which associates uploads with a VikingFile account so they show up in
that account's file list instead of being anonymous/untraceable uploads.
"""

import os
import time
from typing import Dict, List, Optional

import requests
from tqdm import tqdm

from config import get_logger, VIKINGFILE_API_BASE, VIKINGFILE_USER_HASH, MAX_RETRIES, RETRY_DELAY, MAX_RETRY_DELAY

logger = get_logger(__name__)
logger.setLevel(__import__("logging").WARNING)


def _request_with_retry(method: str, url: str, **kwargs) -> requests.Response:
    """Perform an HTTP request with basic exponential-backoff retries."""
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.request(method, url, timeout=kwargs.pop("timeout", 60), **kwargs)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_exc = e
            if attempt < MAX_RETRIES:
                delay = min(RETRY_DELAY * (2 ** (attempt - 1)), MAX_RETRY_DELAY)
                logger.warning(f"Request failed ({e}); retrying in {delay}s (attempt {attempt}/{MAX_RETRIES})")
                time.sleep(delay)
    raise RuntimeError(f"Request to {url} failed after {MAX_RETRIES} attempts: {last_exc}")


def get_upload_url(size: int) -> dict:
    """Request an upload session for a file of the given size (in bytes)."""
    resp = _request_with_retry("POST", f"{VIKINGFILE_API_BASE}/get-upload-url", data={"size": size})
    return resp.json()


def get_upload_server() -> str:
    """Get the legacy single-request upload server URL."""
    resp = _request_with_retry("GET", f"{VIKINGFILE_API_BASE}/get-server")
    return resp.json()["server"]


def complete_upload(key: str, upload_id: str, parts: List[dict], name: str,
                     user: str = "", path: Optional[str] = None) -> dict:
    """Finalize a multi-part upload."""
    data = {"key": key, "uploadId": upload_id, "name": name, "user": user or ""}
    if path:
        data["path"] = path
    for i, part in enumerate(parts):
        data[f"parts[{i}][PartNumber]"] = part["PartNumber"]
        data[f"parts[{i}][ETag]"] = part["ETag"]
    resp = _request_with_retry("POST", f"{VIKINGFILE_API_BASE}/complete-upload", data=data)
    return resp.json()


def list_files(user: str, page: int = 1, path: Optional[str] = None) -> dict:
    """List files already uploaded under an account (and optional path)."""
    data = {"user": user, "page": page}
    if path:
        data["path"] = path
    resp = _request_with_retry("POST", f"{VIKINGFILE_API_BASE}/list-files", data=data)
    return resp.json()


class VikingFileUploader:
    """Uploader that mirrors local files/folders to VikingFile with progress bars
    and duplicate detection, similar in spirit to a cloud-drive sync tool."""

    def __init__(self, user: Optional[str] = VIKINGFILE_USER_HASH, skip_existing: bool = True):
        """
        Args:
            user: VikingFile account user hash. Use None/"" for anonymous uploads
                  (anonymous uploads cannot be listed/deduped or managed later).
            skip_existing: If True, skip files that already exist at the same
                           remote path/name for this account.
        """
        self.user = user or ""
        self.skip_existing = skip_existing
        self._remote_file_cache: Dict[str, Optional[dict]] = {}

    # ---- duplicate detection -------------------------------------------------

    def _remote_files_in_path(self, remote_path: Optional[str]) -> List[dict]:
        if not self.user:
            return []  # anonymous uploads can't be listed
        cache_key = remote_path or ""
        if cache_key in self._remote_file_cache:
            return self._remote_file_cache[cache_key]  # type: ignore

        all_files: List[dict] = []
        page = 1
        while True:
            try:
                result = list_files(self.user, page=page, path=remote_path)
            except Exception as e:
                logger.warning(f"Could not list existing files: {e}")
                break
            all_files.extend(result.get("files", []))
            if page >= result.get("maxPages", 1):
                break
            page += 1

        self._remote_file_cache[cache_key] = all_files
        return all_files

    def file_exists(self, file_name: str, remote_path: Optional[str]) -> Optional[dict]:
        """Check whether a file with this name already exists at remote_path."""
        for f in self._remote_files_in_path(remote_path):
            if f.get("name") == file_name:
                return f
        return None

    # ---- uploading ------------------------------------------------------------

    def upload_file(self, local_path: str, remote_path: Optional[str] = None,
                     _progress_bar=None, _uploaded_size=None) -> Optional[dict]:
        """
        Upload a single file to VikingFile.

        Args:
            local_path: Path to the local file.
            remote_path: Destination folder path on VikingFile, e.g. "SeedUp/Movies".
                         None uploads to the account root.

        Returns:
            Dict with 'name', 'size', 'hash', 'url' on success, None on failure.
        """
        file_name = os.path.basename(local_path)
        file_size = os.path.getsize(local_path)

        if self.skip_existing:
            existing = self.file_exists(file_name, remote_path)
            if existing:
                if _progress_bar is not None and _uploaded_size is not None:
                    _uploaded_size[0] += file_size
                    _progress_bar.update(file_size)
                logger.info(f"Skipping (already exists): {file_name}")
                return {"name": file_name, "size": file_size, "skipped": True,
                         "hash": existing.get("hash"), "url": f"https://vikingfile.com/f/{existing.get('hash')}"}

        try:
            session = get_upload_url(file_size)
            key = session["key"]
            upload_id = session["uploadId"]
            part_size = session["partSize"]
            urls = session["urls"]

            parts = []
            with open(local_path, "rb") as f:
                for part_number, url in enumerate(urls, start=1):
                    chunk = f.read(part_size)
                    if not chunk:
                        break
                    resp = _request_with_retry("PUT", url, data=chunk, timeout=600)
                    etag = resp.headers.get("ETag", "").strip('"')
                    parts.append({"PartNumber": part_number, "ETag": etag})

                    if _progress_bar is not None and _uploaded_size is not None:
                        _uploaded_size[0] += len(chunk)
                        _progress_bar.update(len(chunk))

            result = complete_upload(key, upload_id, parts, file_name, self.user, remote_path)

            # Invalidate the listing cache for this path since it just changed.
            self._remote_file_cache.pop(remote_path or "", None)
            return result

        except Exception as e:
            logger.error(f"Failed to upload '{file_name}': {e}")
            return None

    # ---- helpers for folder uploads --------------------------------------------

    def count_items(self, local_path: str) -> Dict[str, int]:
        """Count total files and total size under a path (file or folder)."""
        if os.path.isfile(local_path):
            return {"files": 1, "total_size": os.path.getsize(local_path)}

        files = 0
        total_size = 0
        for root, _dirs, filenames in os.walk(local_path):
            for filename in filenames:
                try:
                    files += 1
                    total_size += os.path.getsize(os.path.join(root, filename))
                except OSError:
                    pass
        return {"files": files, "total_size": total_size}

    def upload_path(self, local_path: str, remote_base_path: Optional[str] = None) -> Dict[str, list]:
        """
        Upload a file or recursively upload a folder to VikingFile, preserving
        the folder structure as a remote path string (e.g. "SeedUp/MovieName").

        Returns:
            Dict with 'success', 'failed', 'skipped' lists (of local paths) and
            'links' (list of {'local': ..., 'url': ...} for uploaded files).
        """
        results = {"success": [], "failed": [], "skipped": [], "links": []}

        if not os.path.exists(local_path):
            results["failed"].append(local_path)
            return results

        stats = self.count_items(local_path)
        size_mb = stats["total_size"] / (1024 * 1024)
        print(f"📤 Uploading {stats['files']} file(s) ({size_mb:.1f} MB) to VikingFile")

        progress_bar = tqdm(
            total=stats["total_size"],
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Upload",
            bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{rate_fmt}] {postfix}",
            leave=True,
            ncols=100,
        )
        uploaded_size = [0]

        if os.path.isfile(local_path):
            self._upload_single(local_path, remote_base_path, results, progress_bar, uploaded_size)
        else:
            root_name = remote_base_path or os.path.basename(local_path.rstrip(os.sep))
            for dirpath, _dirnames, filenames in os.walk(local_path):
                rel_dir = os.path.relpath(dirpath, local_path)
                if rel_dir == ".":
                    remote_dir = root_name
                else:
                    remote_dir = "/".join([root_name] + rel_dir.split(os.sep))
                for filename in filenames:
                    file_path = os.path.join(dirpath, filename)
                    self._upload_single(file_path, remote_dir, results, progress_bar, uploaded_size)

        progress_bar.close()
        print()
        return results

    def _upload_single(self, local_path, remote_path, results, progress_bar, uploaded_size):
        outcome = self.upload_file(local_path, remote_path,
                                    _progress_bar=progress_bar, _uploaded_size=uploaded_size)
        if outcome is None:
            results["failed"].append(local_path)
        elif outcome.get("skipped"):
            results["skipped"].append(local_path)
            results["links"].append({"local": local_path, "url": outcome.get("url")})
        else:
            results["success"].append(local_path)
            results["links"].append({"local": local_path, "url": outcome.get("url")})

    def print_summary(self, results: Dict[str, list]):
        """Print a clean upload summary with links to each uploaded file."""
        print("\n" + "=" * 60)
        print("🎉 UPLOAD COMPLETE")
        print("=" * 60)
        print(f"✅ {len(results['success'])} file(s) uploaded successfully")

        if results.get("skipped"):
            print(f"⏭️  {len(results['skipped'])} file(s) skipped (already exist)")

        if results["failed"]:
            print(f"❌ {len(results['failed'])} file(s) failed")

        if results.get("links"):
            print("\n📁 Links:")
            for entry in results["links"]:
                name = os.path.basename(entry["local"])
                print(f"   {name} -> {entry['url']}")

        print("=" * 60)


def upload_to_vikingfile(local_path: str, remote_path: Optional[str] = None, **kwargs) -> Dict[str, list]:
    """
    Upload a file or folder to VikingFile.

    Args:
        local_path: Path to file or folder to upload.
        remote_path: Destination folder path on VikingFile (optional). Defaults
                     to a folder named after the uploaded file/folder.
        **kwargs: Additional options:
            - user (str): VikingFile account user hash (defaults to configured hash).
            - skip_existing (bool): Skip files that already exist (default: True).

    Returns:
        Dictionary with 'success', 'failed', 'skipped', and 'links' lists.
    """
    user = kwargs.get("user", VIKINGFILE_USER_HASH)
    skip_existing = kwargs.get("skip_existing", True)

    uploader = VikingFileUploader(user=user, skip_existing=skip_existing)
    results = uploader.upload_path(local_path, remote_path)
    uploader.print_summary(results)
    return results


In [ ]:
%%writefile main.py
#!/usr/bin/env python3
"""
SeedUp - Smart Torrent Management Tool
A Python-based tool that combines torrent downloading with VikingFile uploading capabilities.

Copyright 2025 Ishara Deshapriya

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

Main entry point for torrent downloader with VikingFile upload (Colab-optimized).
Combines torrent downloading and cloud storage capabilities.
"""

import sys
import argparse
import os
from pathlib import Path

from torrent_downloader import download_torrent, download_torrents, get_download_status, clear_session, list_torrent_files
from config import ConfigManager, TORRENT_DOWNLOAD_PATH, VIKINGFILE_USER_HASH, get_logger

logger = get_logger(__name__)


def get_uploader():
    """Import and return uploader function."""
    try:
        from vikingfile_uploader import upload_to_vikingfile
        return upload_to_vikingfile
    except ImportError as e:
        logger.error(f"Failed to import uploader: {str(e)}")
        print("\n" + "="*60)
        print("ERROR: Failed to import VikingFile uploader")
        print("="*60)
        print("Please ensure all required packages are installed:")
        print("  pip install requests tqdm")
        print("="*60)
        raise


def parse_arguments():
    """Parse command line arguments."""
    parser = argparse.ArgumentParser(
        description='Download torrents and upload to VikingFile (Colab-optimized)',
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # List the files inside a torrent (no data downloaded)
  python main.py files -t "magnet:?xt=urn:btih:..."
  
  # Download torrent only
  python main.py download -t movie.torrent
  python main.py download -t "magnet:?xt=urn:btih:..."
  
  # Download only specific files (indices from `files` command)
  python main.py download -t "magnet:?xt=urn:btih:..." --select-files 0,2,5

  # Ranges are supported too
  python main.py download -t "magnet:?xt=urn:btih:..." --select-files 0-3,7
  
  # Download and upload to VikingFile
  python main.py download -t "magnet:?xt=urn:btih:..." --upload -p "SeedUp/Movies"

  # Download multiple torrents at once, in one shared session
  python main.py download-multi -t "magnet:...one" -t "magnet:...two" --upload

  # Same, with per-torrent file selection (line up with -t order; use "" to skip selection for a torrent)
  python main.py download-multi -t "magnet:...one" -t "magnet:...two" -s "0,2" -s ""
  
  # Upload existing files to VikingFile
  python main.py upload -p /path/to/folder
  
  # Upload without skipping existing files
  python main.py upload -p /path --no-skip
  
  # Upload anonymously (ignores the configured user hash)
  python main.py upload -p /path --anonymous
  
  # Check for paused downloads
  python main.py status
  
  # Clear download session
  python main.py clear
  
Note: Uploads are associated with the VikingFile user hash configured in
config.py (VIKINGFILE_USER_HASH) unless --anonymous is passed.
        """
    )
    
    subparsers = parser.add_subparsers(dest='command', help='Command to execute')
    
    # Download command
    download_parser = subparsers.add_parser('download', help='Download a torrent')
    download_parser.add_argument(
        '-t', '--torrent',
        type=str,
        required=True,
        help='Torrent file path or magnet link'
    )
    download_parser.add_argument(
        '-d', '--destination',
        type=str,
        default=TORRENT_DOWNLOAD_PATH,
        help=f'Download destination (default: {TORRENT_DOWNLOAD_PATH})'
    )
    download_parser.add_argument(
        '--no-resume',
        action='store_true',
        help='Start fresh download (ignore previous session)'
    )
    download_parser.add_argument(
        '--upload',
        action='store_true',
        help='Upload to VikingFile after download'
    )
    download_parser.add_argument(
        '-p', '--path',
        type=str,
        help='VikingFile destination folder path (optional, e.g. "SeedUp/Movies")'
    )
    download_parser.add_argument(
        '--no-skip',
        action='store_true',
        help='Force re-upload even if a file with the same name exists remotely'
    )
    download_parser.add_argument(
        '--anonymous',
        action='store_true',
        help='Upload anonymously instead of using the configured user hash'
    )
    download_parser.add_argument(
        '--select-files',
        type=str,
        help='File indices to download (from the `files` command), e.g. "0,2,5" '
             'or with ranges "0-3,7". Omit to download every file in the torrent.'
    )

    # Files command (preview a torrent's contents without downloading)
    files_parser = subparsers.add_parser('files', help="List a torrent's files without downloading any data")
    files_parser.add_argument(
        '-t', '--torrent',
        type=str,
        required=True,
        help='Torrent file path or magnet link'
    )

    # Download-multi command (concurrent multi-torrent download in one session)
    multi_parser = subparsers.add_parser(
        'download-multi',
        help='Download multiple torrents concurrently in a single shared session'
    )
    multi_parser.add_argument(
        '-t', '--torrent',
        action='append',
        required=True,
        help='Torrent file path or magnet link. Repeat -t once per torrent.'
    )
    multi_parser.add_argument(
        '-s', '--select-files',
        action='append',
        default=None,
        help='File indices for the -t entry at the same position (e.g. "0,2,5" or "0-3,7"). '
             'Use "" for a torrent you want downloaded in full. Optional — omit entirely to '
             'download every file in every torrent.'
    )
    multi_parser.add_argument(
        '-d', '--destination',
        type=str,
        default=TORRENT_DOWNLOAD_PATH,
        help=f'Shared download destination; each torrent gets its own subfolder here (default: {TORRENT_DOWNLOAD_PATH})'
    )
    multi_parser.add_argument(
        '--no-resume',
        action='store_true',
        help='Start fresh (ignore previous session)'
    )
    multi_parser.add_argument(
        '--upload',
        action='store_true',
        help='Upload each torrent to VikingFile as soon as the whole batch finishes downloading'
    )
    multi_parser.add_argument(
        '-p', '--path',
        type=str,
        help='Shared VikingFile destination base folder path. Each torrent uploads to '
             '"<path>/<torrent name>" so they don\'t collide. Omit to auto-name each one.'
    )
    multi_parser.add_argument(
        '--no-skip',
        action='store_true',
        help='Force re-upload even if a file with the same name exists remotely'
    )
    multi_parser.add_argument(
        '--anonymous',
        action='store_true',
        help='Upload anonymously instead of using the configured user hash'
    )

    # Upload command
    upload_parser = subparsers.add_parser('upload', help='Upload files to VikingFile')
    upload_parser.add_argument(
        '-p', '--path',
        type=str,
        required=True,
        help='Local path to file or folder to upload'
    )
    upload_parser.add_argument(
        '-r', '--remote-path',
        type=str,
        help='VikingFile destination folder path (optional, e.g. "SeedUp/Movies")'
    )
    upload_parser.add_argument(
        '--no-skip',
        action='store_true',
        help='Force re-upload even if a file with the same name exists remotely'
    )
    upload_parser.add_argument(
        '--anonymous',
        action='store_true',
        help='Upload anonymously instead of using the configured user hash'
    )
    
    # Status command
    subparsers.add_parser('status', help='Check download status')
    
    # Clear command
    subparsers.add_parser('clear', help='Clear download session')
    
    return parser.parse_args()


def handle_download(args):
    """Handle torrent download command."""
    print("="*60)
    print("TORRENT DOWNLOADER")
    print("="*60)
    
    # Parse --select-files into a list of indices, if given.
    # Accepts comma-separated indices and/or ranges, e.g. "0,2,5-7".
    file_indices = None
    if args.select_files:
        try:
            file_indices = parse_file_selection(args.select_files)
        except ValueError:
            logger.error(
                f"Invalid --select-files value: {args.select_files!r} "
                "(expected comma-separated indices and/or ranges, e.g. '0,2,5-7')"
            )
            return 1

    # Download the torrent
    logger.info(f"Starting download: {args.torrent}")
    downloaded_path = download_torrent(
        args.torrent,
        download_path=args.destination,
        auto_resume=not args.no_resume,
        file_indices=file_indices
    )
    
    if not downloaded_path:
        logger.error("Download failed or was cancelled")
        return 1
    
    logger.info(f"Download completed: {downloaded_path}")
    
    # Upload to VikingFile if requested
    if args.upload:
        print("\n" + "="*60)
        print("UPLOADING TO VIKINGFILE")
        print("="*60)
        
        try:
            upload_to_vikingfile = get_uploader()
            
            user = "" if args.anonymous else VIKINGFILE_USER_HASH
            results = upload_to_vikingfile(
                downloaded_path,
                args.path,
                user=user,
                skip_existing=not args.no_skip
            )
            
            if results['failed']:
                logger.warning(f"Some files failed to upload ({len(results['failed'])} items)")
                return 1
            
            logger.info("Upload completed successfully!")
            
        except Exception as e:
            logger.error(f"Upload failed: {str(e)}")
            return 1
    
    return 0


def handle_download_multi(args):
    """Handle concurrent multi-torrent download command."""
    print("="*60)
    print("MULTI-TORRENT DOWNLOADER")
    print("="*60)

    sources = args.torrent
    selections = args.select_files or []

    file_indices_map = {}
    for i, source in enumerate(sources):
        raw = selections[i] if i < len(selections) else ""
        if raw:
            try:
                file_indices_map[source] = parse_file_selection(raw)
            except ValueError:
                logger.error(
                    f"Invalid --select-files value for torrent #{i} ({source[:40]}...): {raw!r} "
                    "(expected comma-separated indices and/or ranges, e.g. '0,2,5-7')"
                )
                return 1

    # If --upload is set, each torrent is uploaded to VikingFile the moment
    # IT finishes, while the rest of the batch keeps downloading in the
    # background — rather than waiting for the whole batch to complete first.
    on_complete = None
    upload_state = {"any_failed": False}

    if args.upload:
        try:
            upload_to_vikingfile = get_uploader()
        except Exception as e:
            logger.error(f"Upload failed: {str(e)}")
            return 1

        user = "" if args.anonymous else VIKINGFILE_USER_HASH

        def on_complete(source, downloaded_path):
            name = os.path.basename(downloaded_path.rstrip(os.sep))
            remote_path = f"{args.path}/{name}" if args.path else None
            try:
                upload_results = upload_to_vikingfile(
                    downloaded_path,
                    remote_path,
                    user=user,
                    skip_existing=not args.no_skip
                )
                if upload_results['failed']:
                    upload_state["any_failed"] = True
                    return f"⚠️ {name}: uploaded with some failures"
                return f"📤 {name}: uploaded to VikingFile"
            except Exception as e:
                logger.error(f"Upload failed for '{downloaded_path}': {str(e)}")
                upload_state["any_failed"] = True
                return f"❌ {name}: upload error ({e})"

    logger.info(f"Starting batch download of {len(sources)} torrent(s)")
    results = download_torrents(
        sources,
        download_path=args.destination,
        auto_resume=not args.no_resume,
        file_indices_map=file_indices_map,
        on_torrent_complete=on_complete
    )

    succeeded = {s: p for s, p in results.items() if p}
    failed = [s for s, p in results.items() if not p]

    print("\n" + "="*60)
    print("BATCH DOWNLOAD SUMMARY")
    print("="*60)
    print(f"✅ {len(succeeded)} succeeded, ❌ {len(failed)} failed/incomplete")

    if failed:
        for s in failed:
            print(f"   ❌ {s[:70]}")

    if not succeeded:
        logger.error("No torrents completed successfully")
        return 1

    if args.upload:
        if upload_state["any_failed"]:
            logger.warning("One or more uploads had failures — see log above")
            return 1
        logger.info("All uploads completed successfully!")

    return 0 if not failed else 1


def parse_file_selection(selection: str):
    """
    Parse a file-selection string into a sorted list of unique indices.

    Accepts comma-separated indices and/or inclusive ranges, e.g.:
        "0,2,5"      -> [0, 2, 5]
        "0-3,7"      -> [0, 1, 2, 3, 7]
        "0-3,5-7"    -> [0, 1, 2, 3, 5, 6, 7]

    Raises ValueError on malformed input.
    """
    indices = set()
    for part in selection.split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            start_str, end_str = part.split('-', 1)
            start, end = int(start_str.strip()), int(end_str.strip())
            if start > end:
                start, end = end, start
            indices.update(range(start, end + 1))
        else:
            indices.add(int(part))
    return sorted(indices)


def _format_size(num_bytes):
    """Human-readable file size."""
    size = float(num_bytes)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} PB"


def handle_files(args):
    """Handle listing a torrent's files without downloading."""
    print("="*60)
    print("TORRENT FILE LIST")
    print("="*60)

    logger.info(f"Fetching metadata: {args.torrent}")
    files = list_torrent_files(args.torrent)

    if files is None:
        logger.error("Failed to fetch torrent file list")
        return 1

    print(f"\nFound {len(files)} file(s):\n")
    for f in files:
        print(f"  [{f['index']:>3}] {_format_size(f['size']):>10}   {f['path']}")

    total_size = sum(f['size'] for f in files)
    print(f"\nTotal size (all files): {_format_size(total_size)}")
    print("\nTo download only some of these, use:")
    print(f'  python main.py download -t "{args.torrent}" --select-files <indices, e.g. 0,2,5-7>')
    return 0


def handle_upload(args):
    """Handle VikingFile upload command."""
    print("="*60)
    print("VIKINGFILE UPLOADER")
    print("="*60)
    
    # Validate path exists
    if not os.path.exists(args.path):
        logger.error(f"Path does not exist: {args.path}")
        return 1
    
    # Upload to VikingFile
    try:
        upload_to_vikingfile = get_uploader()
        
        user = "" if args.anonymous else VIKINGFILE_USER_HASH
        results = upload_to_vikingfile(
            args.path,
            args.remote_path,
            user=user,
            skip_existing=not args.no_skip
        )
        
        if results['failed']:
            logger.warning(f"Some files failed to upload ({len(results['failed'])} items)")
            return 1
        
        logger.info("Upload completed successfully!")
        return 0
    
    except Exception as e:
        logger.error(f"Upload failed: {str(e)}")
        return 1


def handle_status(args):
    """Handle status check command."""
    if get_download_status():
        print("✓ Found paused download session")
        print("  Run 'python main.py download -t <torrent>' to resume")
        return 0
    else:
        print("✗ No paused download session found")
        return 0


def handle_clear(args):
    """Handle clear session command."""
    if clear_session():
        print("✓ Download session cleared")
        return 0
    else:
        print("✗ Failed to clear session")
        return 1


def main():
    """Main entry point."""
    args = parse_arguments()
    
    # Show help if no command specified
    if not args.command:
        print("Error: No command specified\n")
        parse_arguments().print_help()
        return 1
    
    try:
        # Route to appropriate handler
        if args.command == 'download':
            return handle_download(args)
        elif args.command == 'files':
            return handle_files(args)
        elif args.command == 'download-multi':
            return handle_download_multi(args)
        elif args.command == 'upload':
            return handle_upload(args)
        elif args.command == 'status':
            return handle_status(args)
        elif args.command == 'clear':
            return handle_clear(args)
        else:
            logger.error(f"Unknown command: {args.command}")
            return 1
            
    except KeyboardInterrupt:
        print("\n\nOperation cancelled by user")
        if args.command == 'download':
            print("Download progress has been saved. Resume with the same command.")
        return 130
    except Exception as e:
        logger.error(f"Operation failed: {str(e)}", exc_info=True)
        return 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile requirements.txt
# Torrent dependencies
libtorrent

# VikingFile upload dependencies
requests

# UI and utilities
tqdm

# Optional: For Google Colab
# google-colab (automatically available in Colab)


In [ ]:
#@title **Verify Project Files** { display-mode: "form" }
import os

required_files = [
    "main.py",
    "config.py",
    "vikingfile_uploader.py",
    "torrent_downloader.py",
    "requirements.txt",
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print("⚠️ Missing files:", ", ".join(missing))
    print("Re-run the file creation cells above.")
else:
    print("✅ All project files created successfully!")


---
# **🔧 Step 2: Install Dependencies**

This will install all required packages including libtorrent and the VikingFile upload dependencies.

In [ ]:
#@title **Install Required Packages** { display-mode: "form" }
#@markdown Click the ▶️ button to install dependencies (takes ~2 minutes)

print("📦 Installing dependencies...\n")

# Install libtorrent
!pip install -r requirements.txt > /dev/null 2>&1


print("\n✅ All dependencies installed successfully!")
print("📌 Ready to proceed to the next step.")

In [ ]:
#@title **Verify Installations** { display-mode: "form" }
#@markdown Click the ▶️ button to verify all installations are working properly

import sys
import subprocess

def verify_installation():
    print("🔍 Verifying installations...\n")

    # Check Python version
    py_version = sys.version.split()[0]
    print(f"📌 Python Version: {py_version}")

    # Check libtorrent
    try:
        import libtorrent as lt
        print(f"📌 libtorrent Version: {lt.version}")
    except ImportError:
        print("❌ libtorrent not found!")
        return False

    # Check VikingFile upload dependencies
    try:
        import requests
        print("📌 requests library: ✓")
    except ImportError:
        print("❌ requests library not found!")
        return False

    print("\n✅ All dependencies verified successfully!")
    return True

verify_installation()

# **🔐 Step 3: Configure VikingFile**

VikingFile uploads use a simple account "user hash" instead of an OAuth login popup.

In [ ]:
#@title **Configure VikingFile Account (Optional)** { display-mode: "form" }
#@markdown VikingFile doesn't use a login popup like Google Drive. Instead, set
#@markdown your account's user hash directly in `config.py`
#@markdown (`VIKINGFILE_USER_HASH = "..."`) so uploads are associated with your
#@markdown account. Leave it blank to upload anonymously.

with open("config.py") as f:
    contents = f.read()

import re
match = re.search(r'VIKINGFILE_USER_HASH\s*=\s*"([^"]*)"', contents)
current_hash = match.group(1) if match else "(not set)"

print(f"📌 Current configured VikingFile user hash: {current_hash}")
print("📌 Edit config.py directly to change it.")


# **🗂️ Step 4: List Torrent Files (Optional)**

Preview which files are inside a torrent, without downloading any data. Use the printed index numbers to only download specific files in the next step (useful for multi-file torrents where you don't want everything).

In [ ]:
#@title **List Torrent Files** { display-mode: "form" }

#@markdown Paste magnet link OR path to .torrent file:
TORRENT_SOURCE_PREVIEW = "" #@param {type:"string"}

if not TORRENT_SOURCE_PREVIEW:
    print("⚠️ Please enter a magnet link or torrent file path!")
else:
    !python main.py files -t "{TORRENT_SOURCE_PREVIEW}"


# **🌐 Step 5: Download Torrent**

Download a torrent using either:
- **Magnet Link:** `magnet:?xt=urn:btih:...`
- **Torrent File:** Upload a .torrent file to Colab first

### Options:
- **Auto Upload:** Automatically upload to VikingFile after download
- **Skip Existing:** Skip files that already exist remotely

**Note:** Files will be uploaded to a folder named after the download automatically (or your custom folder path if specified)

In [ ]:
#@title **Download Torrent** { display-mode: "form" }

#@markdown ### 🔗 Enter Torrent Information
#@markdown Paste magnet link OR path to .torrent file:
TORRENT_SOURCE = "" #@param {type:"string"}

#@markdown ### 🗂️ File Selection (optional)
#@markdown File indices from the "List Torrent Files" step above, e.g. "0,2,5"
#@markdown or with ranges "0-3,7". Leave blank to download every file in the torrent.
SELECT_FILES = "" #@param {type:"string"}

#@markdown ### ⚙️ Options
AUTO_UPLOAD = True #@param {type:"boolean"}
SKIP_EXISTING = True #@param {type:"boolean"}
#@markdown Destination folder path on VikingFile (optional, e.g. "SeedUp/Movies"):
VIKINGFILE_PATH = "" #@param {type:"string"}

import sys

if not TORRENT_SOURCE:
    print("⚠️ Please enter a magnet link or torrent file path!")
else:
    print("🚀 Starting torrent download...\n")

    if SELECT_FILES:
        print(f"🗂️ Only downloading selected file indices: {SELECT_FILES}\n")

    if AUTO_UPLOAD:
        dest = VIKINGFILE_PATH if VIKINGFILE_PATH else "(auto folder name)"
        print(f"📁 Files will be uploaded to VikingFile: {dest}\n")

    # Build command
    cmd = f'python main.py download -t "{TORRENT_SOURCE}"'

    if SELECT_FILES:
        cmd += f' --select-files {SELECT_FILES}'

    if AUTO_UPLOAD:
        cmd += " --upload"
        if VIKINGFILE_PATH:
            cmd += f' -p "{VIKINGFILE_PATH}"'

    if not SKIP_EXISTING:
        cmd += " --no-skip"

    # Execute download
    !{cmd}


# **🌐📦 Step 5b: Download Multiple Torrents at Once**

Downloads several torrents concurrently in a single shared session, with combined progress for all of them, and uploads each one to VikingFile the moment IT finishes (not waiting for the whole batch) if Auto Upload is on.

Separate multiple magnet links/torrent paths with ` | ` (a pipe, with spaces around it).

`SELECT_FILES` is optional and lines up with `TORRENT_SOURCES` by position, also `|`-separated: use indices/ranges (e.g. `0,2,5-7`) from Step 4 for a torrent, or leave that slot blank to download that torrent in full.

In [ ]:
#@title **Download Multiple Torrents** { display-mode: "form" }

#@markdown Separate multiple magnet links / torrent paths with " | " (a pipe, with spaces around it):
TORRENT_SOURCES = "" #@param {type:"string"}

#@markdown Optional: file selection per torrent above, same order, also " | "-separated.
#@markdown Leave a slot blank to download that torrent in full, e.g.: 0,2,5 |  | 0-3,7
SELECT_FILES = "" #@param {type:"string"}

#@markdown ### ⚙️ Options
AUTO_UPLOAD = True #@param {type:"boolean"}
SKIP_EXISTING = True #@param {type:"boolean"}
#@markdown Shared VikingFile destination base folder (optional). Each torrent
#@markdown uploads to "<this>/<torrent name>" so they don't collide:
VIKINGFILE_PATH = "" #@param {type:"string"}

sources = [s.strip() for s in TORRENT_SOURCES.split("|") if s.strip()]
selections = [s.strip() for s in SELECT_FILES.split("|")]

if not sources:
    print("⚠️ Add at least one magnet link or torrent path to TORRENT_SOURCES!")
else:
    print(f"🚀 Starting batch download of {len(sources)} torrent(s)...\n")

    cmd = "python main.py download-multi"
    for src in sources:
        cmd += f' -t "{src}"'
    for i in range(len(sources)):
        sel = selections[i] if i < len(selections) else ""
        cmd += f' -s "{sel}"'

    if AUTO_UPLOAD:
        cmd += " --upload"
        if VIKINGFILE_PATH:
            cmd += f' -p "{VIKINGFILE_PATH}"'

    if not SKIP_EXISTING:
        cmd += " --no-skip"

    !{cmd}


# **📤 Step 6: Upload Existing Files**

If you already have downloaded files and want to upload them to VikingFile separately.

**Note:** Files will be uploaded to a folder named after the download automatically (or your custom folder path if specified)

In [ ]:
#@title **Upload Files to VikingFile** { display-mode: "form" }

#@markdown ### 📁 File/Folder to Upload
LOCAL_PATH = "" #@param {type:"string"}

#@markdown ### ⚙️ Options
SKIP_EXISTING_FILES = True #@param {type:"boolean"}
#@markdown Destination folder path on VikingFile (optional, e.g. "SeedUp/Movies"):
VIKINGFILE_PATH = "" #@param {type:"string"}

import os

if not LOCAL_PATH:
    print("⚠️ Please enter a file or folder path to upload!")
elif not os.path.exists(LOCAL_PATH):
    print(f"⚠️ Path does not exist: {LOCAL_PATH}")
else:
    print("📤 Starting upload to VikingFile...\n")

    # Build command
    cmd = f'python main.py upload -p "{LOCAL_PATH}"'

    if VIKINGFILE_PATH:
        cmd += f' -r "{VIKINGFILE_PATH}"'

    if not SKIP_EXISTING_FILES:
        cmd += " --no-skip"

    # Execute upload
    !{cmd}


# **🔍 Step 7: Check Download Status**

Check if there's a paused download that can be resumed.

In [ ]:
#@title **Check Status** { display-mode: "form" }

!python main.py status

# **🧹 Step 8: Clear Session (Optional)**

Clear any saved download session if you want to start fresh.

In [ ]:
#@title **Clear Download Session** { display-mode: "form" }

!python main.py clear


---

## 📚 **Additional Information**

### Troubleshooting

**Download is slow:**
- This depends on the number of seeders and your torrent health
- Colab's network speed varies
- Try using different trackers or magnet links

**Upload failing:**
- Check your internet connection is stable
- Double-check your user hash in config.py if uploads aren't appearing in your account
- Check if files already exist remotely when using the skip option
- Destination folders on VikingFile are created automatically as part of the upload path

**Session disconnected:**
- Downloads are saved and can be resumed
- Use the status check to verify saved sessions
- Run the download command again with the same torrent
- Clear stuck sessions if needed

### Best Practices

1. **For large downloads:**
   - Monitor the session and be ready to resume if it disconnects
   - Use the keep-alive cell to prevent idle disconnects
   - Split very large downloads into smaller parts

2. **Storage management:**
   - All files are organized in the 'SeedUp Downloads' folder automatically
   - Clear the downloads folder after successful upload
   - Monitor Colab's disk space usage
   - Use `skip-existing` to avoid duplicate uploads

3. **Finding your uploads:**
   - Each uploaded file's `https://vikingfile.com/f/<hash>` link is printed after each upload
   - If you configured a user hash, uploads also appear in your VikingFile account's file list

### Session Management

The project uses a sophisticated session management system that:
- Saves download progress automatically
- Enables resuming from the exact point of interruption
- Maintains tracker and peer information
- Cleans up automatically after successful completion

---
<br>

### 🔗 **Links**

- [GitHub Repository](https://github.com/codercyco/SeedUp)
- [Report Issues](https://github.com/codercyco/SeedUp/issues)

---

<center>
<p>Made with ❤️ for the community</p>
<p>Created by <b>Ishara Deshapriya</b></p>
<p>Licensed under <b>Apache License 2.0</b></p>
</center>